In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# names.txt is downloaded from https://github.com/karpathy/makemore/blob/master/names.txt
words = open('../data/makemore/names.txt', 'r').read().splitlines()
words[:10]

In [ ]:
len(words)

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
print(itos)

In [ ]:
# # build the dataset

# block_size = 3  # context length
# X, Y = [], []
# for w in words:  # Or set the [:5] to look at the examples and check mini-batch overfitting
#     # print(w)
#     context = [0] * block_size
#     for ch in w + '.':
#         ix = stoi[ch]
#         X.append(context)
#         Y.append(ix)
#         # print(''.join(itos[i] for i in context), '--->', itos[ix])
#         context = context[1:] + [ix]  # Shift the context window

# X = torch.tensor(X)
# Y = torch.tensor(Y)

In [ ]:
def build_dataset(words):
    block_size = 5  # context length
    X, Y = [], []
    for w in words:  # Or set the [:5] to look at the examples and check mini-batch overfitting
        # print(w)
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            # print(''.join(itos[i] for i in context), '--->', itos[ix])
            context = context[1:] + [ix]  # Shift the context window

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

Bellow is the implementation of the [paper](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf) according to the [README.md](https://github.com/karpathy/makemore/blob/master/README.md) of the Karpathy's [makemore](https://github.com/karpathy/makemore) repository.

In [ ]:
# Simple embedding extraction (will be used bellow)
# C[5]  # Extract the embedding for the symbol with the 5-th index

In [ ]:
# Canonical enbedding extraction (will not be used for simplicity)
# F.one_hot(torch.tensor(5), num_classes=27).float() @ C  # Extract the embedding using one-hot vector and matrix mul

In [ ]:
Xtr.shape, Ytr.shape

In [ ]:
# Defining the parameters
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 10), generator=g)
W1 = torch.randn((50, 300), generator=g)  # We will have 3 emb vectos with length 2-10, shape after concatenation: (1, 6-30)
b1 = torch.randn(300, generator=g)
W2 = torch.randn((300, 27), generator=g)  # The output layer, so we want 27 possible characters as the output
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [ ]:
sum(p.nelement() for p in parameters)

In [ ]:
for p in parameters:
    p.requires_grad = True

In [ ]:
# Define the learning rates from the 'barely changes' to 'explodes'
# Generate 1000 learning rates in exponential scale - from 0.001 to 1
# This will be used in mini-batch experiments to define the initial learning rate
# lre = torch.linspace(-3, 0, 1000)
# lrs = 10**lre

In [ ]:
# lri = []
lossi = []
stepi = []

In [ ]:
for i in range(300000):
    # Mini-batch construct
    ix = torch.randint(0, Xtr.shape[0], (64,))  # Batch size is 64

    # Forward pass
    emb = C[Xtr[ix]]  # Extract all the enbeddings at once passing the input tensor (N, 3, 2)
    h = torch.tanh(emb.view(-1, 50) @ W1 + b1)
    logits = h @ W2 + b2  # (N, 27)
    loss = F.cross_entropy(logits, Ytr[ix])
    # print(loss.item())

    # Backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # Use lre above to determine the better learning rate

    # lr = lrs[i]
    lr = 0.1 if i < 200000 else 0.01
    # Parameters update
    for p in parameters:
        p.data += -lr * p.grad

    # lri.append(lre[i])
    stepi.append(i)
    lossi.append(loss.log10().item())

print(loss.item())

In [ ]:
plt.plot(stepi, lossi)

In [ ]:
# Find the optimal point on the plot bellow
# plt.plot(lri, lossi)

In [ ]:
# Here we can check overfitting on the mini batch (exact 0 loss is not possible in this case)
# print(logits.max(1))
# print(Y)

In [ ]:
# Evaluate the model on the train dataset
emb = C[Xtr]
h = torch.tanh(emb.view(-1, 50) @ W1 + b1)
logits = h @ W2 + b2  # (N, 27)
loss = F.cross_entropy(logits, Ytr)
loss

In [ ]:
# Evaluate the model on the dev/validation dataset
emb = C[Xdev]
h = torch.tanh(emb.view(-1, 50) @ W1 + b1)
logits = h @ W2 + b2  # (N, 27)
loss = F.cross_entropy(logits, Ydev)
loss

When the training and dev/validation losses are rawly equal, it typically means 'underfitting' (the size of the model cannot allow to memorize all of the training data), so we need to make the model more complex.

In [ ]:
# Visualization of the embeddings, when they are 2D
# plt.figure(figsize=(8, 8))
# plt.scatter(C[:, 0].data, C[:, 1].data, s=200)
# for i in range(C.shape[0]):
#     plt.text(C[i, 0].item(), C[i, 1].item(), itos[i], ha='center', va='center', color='white')
# plt.grid('minor')

In [ ]:
# Sampling
block_size = 5  # The same as above
inf_gen = torch.Generator().manual_seed(2147483647 + 10)
for i in range(20):
    out = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, 1)

        idx = torch.multinomial(probs, num_samples=1, replacement=True, generator=inf_gen).item()
        context = context[1:] + [idx]
        out.append(itos[idx])
        if idx == 0:
            break

    print(''.join(out).strip('.'))